In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
import requests
from scipy.stats import norm
import matplotlib.pyplot as plt

# ==========================
# USER PARAMETERS
# ==========================
capital = 100000
max_positions = 5
max_allocation_pct = 0.10
simulation_runs = 1000
expected_return = 0.15
annual_volatility = 0.25

# ==========================
# F&O STOCKS ONLY (FASTER)
# ==========================
tickers = [
    "RELIANCE.NS","HDFCBANK.NS","ICICIBANK.NS","INFY.NS","TCS.NS",
    "SBIN.NS","LT.NS","AXISBANK.NS","ITC.NS","BHARTIARTL.NS",
    "KOTAKBANK.NS","BAJFINANCE.NS","MARUTI.NS","TATAMOTORS.NS"
]

# ==========================
# BLACK-SCHOLES
# ==========================
def black_scholes(S, K, T, r, sigma, option_type="call"):
    if S <= 0 or K <= 0 or T <= 0 or sigma <= 0:
        return 0

    d1 = (np.log(S/K) + (r + sigma**2/2)*T) / (sigma*np.sqrt(T))
    d2 = d1 - sigma*np.sqrt(T)

    if option_type == "call":
        return S*norm.cdf(d1) - K*np.exp(-r*T)*norm.cdf(d2)
    else:
        return K*np.exp(-r*T)*norm.cdf(-d2) - S*norm.cdf(-d1)

# ==========================
# RISK FREE RATE
# ==========================
def get_risk_free_rate():
    try:
        bond = yf.download("^INDI10Y", period="5d", progress=False)
        return float(bond["Close"].iloc[-1]) / 100
    except:
        return 0.07

risk_free_rate = get_risk_free_rate()

# ==========================
# GRAHAM VALUE
# ==========================
def graham_value(ticker):
    try:
        stock = yf.Ticker(ticker)
        info = stock.fast_info
        price = info.get("lastPrice")

        full = stock.info
        eps = full.get("trailingEps")

        if eps is None:
            return None, None

        growth = full.get("earningsQuarterlyGrowth")
        if growth is None:
            growth = 0.05

        g = min(growth * 100, 20)

        intrinsic = eps * (8.5 + 2 * g)

        return intrinsic, price

    except:
        return None, None

# ==========================
# NSE SESSION
# ==========================
session = requests.Session()

headers = {
    "User-Agent": "Mozilla/5.0",
    "Accept-Language": "en-US,en;q=0.9"
}

session.get("https://www.nseindia.com", headers=headers, timeout=5)

# ==========================
# OPTION CHAIN
# ==========================
def get_option_chain(symbol):
    try:
        url = f"https://www.nseindia.com/api/option-chain-equities?symbol={symbol}"
        response = session.get(url, headers=headers, timeout=5)
        return response.json()
    except:
        return None

# ==========================
# BEST OPTION TRADE
# ==========================
def find_best_trade(ticker):
    symbol = ticker.replace(".NS", "")
    data = get_option_chain(symbol)

    if data is None:
        return None

    best = None
    best_edge = 0

    try:
        spot = data["records"]["underlyingValue"]

        for row in data["records"]["data"]:
            strike = row["strikePrice"]

            for side in ["CE", "PE"]:
                if side not in row:
                    continue

                opt = row[side]

                premium = opt.get("lastPrice")
                iv = opt.get("impliedVolatility")
                lot = opt.get("marketLot", 1)

                if not premium or not iv:
                    continue

                expiry = pd.to_datetime(opt["expiryDate"])
                days = (expiry - pd.Timestamp.today()).days

                if days <= 0:
                    continue

                T = days / 365
                sigma = iv / 100
                option_type = "call" if side == "CE" else "put"

                theoretical = black_scholes(
                    spot, strike, T, risk_free_rate, sigma, option_type
                )

                edge = theoretical - premium

                if edge > best_edge:
                    max_alloc = capital * max_allocation_pct
                    cost_per_lot = premium * lot
                    lots = int(max_alloc // cost_per_lot)

                    if lots < 1:
                        continue

                    best_edge = edge

                    best = {
                        "Ticker": ticker,
                        "Type": option_type,
                        "Strike": strike,
                        "Expiry": expiry.date(),
                        "Spot": spot,
                        "Premium": premium,
                        "Theoretical": theoretical,
                        "Edge": edge,
                        "Lot Size": lot,
                        "Lots": lots,
                        "Capital Used": lots * cost_per_lot
                    }

    except:
        return None

    return best

# ==========================
# SCAN MARKET
# ==========================
results = []

for ticker in tickers:
    print("Scanning:", ticker)

    intrinsic, price = graham_value(ticker)

    if intrinsic is None or price is None:
        continue

    if price >= intrinsic:
        continue

    trade = find_best_trade(ticker)

    if trade:
        trade["Intrinsic"] = intrinsic
        trade["Undervaluation %"] = ((intrinsic - price) / intrinsic) * 100
        results.append(trade)

# ==========================
# RESULTS
# ==========================
df = pd.DataFrame(results)

if not df.empty:
    df = df.sort_values("Edge", ascending=False).head(max_positions)

    df["Expected Profit"] = (
        (df["Theoretical"] - df["Premium"])
        * df["Lot Size"]
        * df["Lots"]
    )

    print(df)

    try:
        df.to_excel("option_arbitrage_results.xlsx", index=False)
        print("Saved to option_arbitrage_results.xlsx")
    except:
        df.to_csv("option_arbitrage_results.csv", index=False)
        print("Saved to option_arbitrage_results.csv")

else:
    print("No opportunities found.")

# ==========================
# MONTE CARLO
# ==========================
simulated = []

for _ in range(simulation_runs):
    final = capital

    for _ in range(30):
        daily = np.random.normal(
            expected_return / 252,
            annual_volatility / np.sqrt(252)
        )
        final *= (1 + daily)

    simulated.append(final)

simulated = np.array(simulated)

print("\nWorst Case:", np.percentile(simulated, 5))
print("Expected:", np.mean(simulated))
print("Best Case:", np.percentile(simulated, 95))
print("Loss Probability:", np.mean(simulated < capital))

plt.figure()
plt.hist(simulated, bins=40)
plt.title("Monte Carlo Capital Distribution")
plt.xlabel("Final Capital")
plt.ylabel("Frequency")
plt.show()